# ATLAS *Z*+jets Omnifold: Basic usage and setup

This notebook serves as an introduction to interacting with the ATLAS full-phase space measurement of $Z$+jets production in $pp$ collisions at $\sqrt{s}=13$ TeV.
It is based on the notebooks written for the previous [Multifold $Z$+jets measurement][https://gitlab.cern.ch/atlas-physics/public/sm-z-jets-omnifold-2024].
While the previous measurement targeted 24 observables, the full-phase space measurement is differential in the kinematics of every charged particle, making it a variable dimensional measurement.
For this reason the previously used HDF5 files are replaced with ROOT files and read using an uproot/awkward array interface.
Beyond this the introductory methods shown here are very similar.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports
import os
import numpy as np
import uproot
import pandas as pd
import utils.common_utils as cu
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

## Data and weights

The following data files are needed to construct and validate the measurement.
All are provided in the ROOT file format:

- Default Monte-Carlo (MadGraph) sample: `ZjetOmnifold_5Jul2025_MGPy8FxFxPlusNonStrong_syst_Test_withdd.root`
- Alternative Monte-Carlo (Sherpa) sample: `ZjetOmnifold_Mar10_Sherpa2211_LookLike_MgFxFx_Test_V5.root`

In addition, particle-level generator predictions for the MadGraph+Pythia and Sherpa samples are provided in the following locations:

- MadGraph+Pythia: To be added
- Sherpa: To be added

### Data download

To download all of the ROOT files above, run the cell below.
Note this will take a few GB of disk space (**TODO** add exact number)!

In [ ]:
### Download data (reconfigure once data are public)
raise NotImplementedError("Data download not implemented")

### Load and examine data

Once the data are downloaded, run the cells below to load it into memory.
In this notebook, we will only look at the total cross section predicted by the data measurement.
For this, we will need the default MC and Sherpa MC files.

In [ ]:
### Default: load the full measurement
n_events = None

### If running on Binder, which has a memory constraint of 2 GB, uncomment the following line to cut the size of the datasets approximately in half. 
### The results will not be identical to the ones presented in our paper, but can be used for demo & educational purposes.
# n_events = 200_000

In [ ]:
data_dir = "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data"
f = uproot.open(os.path.join(data_dir, "truth_mc_events.root"))
t = f["OmniTree"]
f_sherpa = uproot.open(os.path.join(data_dir, "truth_mc_hv_events.root"))
t_sherpa = f_sherpa["OmniTree"]

In [ ]:
print(f"Examining the branches of the default MC TTree with {t.num_entries} entries")
t.show(name_width=40)

The nominal MC TTree has $1035835$ events, described by many branches. The most important ones are as follows:

- `truth_pT_tracks`: The $p_T$ of all tracks in the events. This is a variable length (jagged) array with different numbers of entries per event.
- `truth_eta_tracks`: The same as above but giving the $\eta$ coordinate
- `truth_phi_tracks`: The same as above but giving the $\phi$ coordinate
- `truth_pdgId_tracks`: An integer designating the PDG ID of the truth charged hadron
- `truth_pT_l1`: The $p_T$ of the leading muon
- `truth_pT_l2`: The $p_T$ of the sub-leading muon
- `truth_eta_l1`: The $\eta$ of the leading muon
- `truth_eta_l2`: The $\eta$ of the sub-leading muon
- `truth_phi_l1`: The $\phi$ of the leading muon
- `truth_phi_l2`: The $\phi$ of the sub-leading muon

These branches are the raw ingredients that can be used to calculate many different observables, all of which will be constrained by this measurement.
However in this notebook, we are only interested in the total cross section which is only a function of the weights:

In [ ]:
print(f"The Sherpa MC TTree has {t_sherpa.num_entries} events")

The Sherpa TTree has $1377524$ events, but beyond this has identical branch names as the nominal TTree.

### Load and examine event weights

Next we will load the event weights that produce the data measurement.
Note that to produce differential cross section measurements, it is recommended to validate using the pseudodata measurement before running the data measurement, but we will only look at the inclusive cross section measurement here.
See the usage recommendations and [standard_obs_pseudo_results.ipynb](./standard_obs_pseudo_results.ipynb) for more details.

Since the event weights are fixed dimensional, they are stored separately in HDF5 files rather than in ROOT files.
There are two HDF5 files needed to build the measurement, one containing event weights for the nominal MadGraph+Pythia MC sample and the other containing weights for the alternative Sherpa sample.
Each of the HDF5 files contains a single pandas dataframe:

In [ ]:
of_data = pd.read_hdf(os.path.join(data_dir, "data_weights.h5"), mode="r")
of_data_hv = pd.read_hdf(os.path.join(data_dir, "data_hv_weights.h5"), mode="r")

In [ ]:
of_data

In [ ]:
of_data_hv

The pandas dataframe has a number of rows equal to the number of events in the corresponding MC sample.
The dataframe for the nominal MC sample has 164 columns, each corresponding to a vector of event weights, while the dataframe for the alternative MC sample has only one vector of weights.
Some accounting for these 164 event weights is as follows:

In [ ]:
def printWeightCounts(dataset,desc):
    columns = np.array(dataset.keys())
    NmcBS=NdataBS=Nsys=Nens=Nprior=Nnom=0
    for column in columns:
        if "bootstrap_mc" in column:
            NmcBS+=1
        elif "bootstrap_data" in column:
            NdataBS+=1
        elif "ensemble" in column:
            Nens+=1
        elif "prior" in column:
            Nprior+=1
        elif "nominal" in column:
            Nnom+=1
        elif "weight" in column:
            Nsys+=1
    print("\nFor {}, there are {} weight vectors:".format(desc,len(columns)))
    print(f"  {Nprior} prior weights, {Nnom} nominal weights, {Nsys} systematic weights, {Nens} NN ensemble weights, {NmcBS} MC bootstrap weights, {NdataBS} data bootstrap weights")

printWeightCounts(of_data,"the nominal MC dataframe")
printWeightCounts(of_data_hv,"the alternative MC dataframe")

From the numbers displayed above, we can see that the nominal MC dataframe contains six categories of weights:

1. **Prior weights**: One vector of weights for the nominal MC sample that gives the prior used as input to the unfolding
2. **Nominal weights**: One vector of weights for the nominal MC sample that gives the central value result for the pseudodata measurement
3. **Systematic weights**: 26 vectors of weights for the nominal MC and 1 vector of weights for the alternative MC that are used to assess systematic uncertainties on the measurement
4. **Ensemble weights**: 10 vectors of weights for the nominal MC that provide the nominal weight predictions over 10 parallel runs of Omnifold. These are used to set the NN initialization uncertainty
5. **MC Bootstrap weights**: 25 vectors of weights for the nominal MC that provide the Omnifold predictions under 25 bootstrap fluctuations of the MC **train** set used for the training of networks. Used to set the MC train statistical uncertainty.
6. **Data Bootstrap weights**: 100 vectors of weights for the nominal MC that provide the Omnifold predictions under 100 bootstrap fluctuations of the data sample. Used to set the data statistical uncertainty.

All of these weights will be used below to obtain a measurement of the inclusive cross section.

## Obtaining measured cross sections, uncertainties and MC predictions

As described in the paper, the measurement is performed in a fiducial region defined by two opposite charge, prompt muons (prompt meaning that they do not originate from a hadron decay) that each fulfill $p_\mathrm{T}>20$ GeV and $|\eta|<2.5$.
The dimuon system is further required to fulfill $m_{\mu\mu} \in (81,101)\,\text{GeV}$ and $p_\text{T}^{\mu\mu}>200\,\text{GeV}$, i.e. be consistent with a boosted $Z$ boson decaying to muons.
Summing up `weights_nominal` of all events in the sample will return the central value of the measured cross section of the full fiducial volume.
One can further construct a subregion based on any combination of selection criteria using any observable constructed from the muon and track kinematics (with certain caveats as discussed below) and obtain associated measured (and predicted) cross sections from the sum of weights of the events that fall in this region.

The central value in a subregion $A$ is given by the sum of nominal weights, `weights_nominal`, in the code, which has units femtobarn:

$$
\hat{\sigma}_A = \sum_{i\in A} w^\text{nom}_i.
$$

A systematically varied cross section corresponding to a nuisance parameter $(k)$ can be obtained using any of the alternative such weights:

$$
\hat{\sigma}_A^{(k)} = \sum_{i\in A} w^{(k)}_i.
$$

From such a variation, the absolute uncertainty amplitude would be given by the differnece to the central value: $\Delta_{A}^{(k)} = \hat{\sigma}_{A}^{(k)} - \hat{\sigma}_{A}$, and these variations should be treated as uncorrelated between each other, and fully correlated between regions (equivalently, bins in differential measurements).
As a consequence, the covariance between two regions $A$ and $B$ from a given nuisance parameter $k$ is evaluated as $V_{A,B}^{(k)}=\Delta_{A}^{(k)}\,\Delta_{B}^{(k)}$, and the covariance from several such NPs is given by the sum: $V_{A,B} = \sum_k V_{A,B}^{(k)}$.

Statistical uncertainties are evaluated using bootstrap weights. The statistical covariance between two regions $A$ and $B$ is

$$
V^\text{stat}_{A,B} = \frac{1}{N_\text{BS}}\sum_{b=1}^{N_\mathrm{BS}}(\hat{\sigma}^{(b)}_A-\hat{\sigma}^\text{nom}_A)(\hat{\sigma}^{(b)}_B-\hat{\sigma}^\text{nom}_B).
$$

The code blocks below provide examples of how to extract the results needed to perform a cross section measurement in a particular kinematic region (one bin).
For examples of how to perform differential measurements and construct the full covariance matrix, see the standard observable notebooks [standard_obs_pseudo_results.ipynb](./standard_obs_pseudo_results.ipynb) amd [standard_obs_data_results.ipynb](./standard_obs_data_results.ipynb), as well as the `UncertaintyCalculator` class in [uncertainties.py](./uncertainties.py).

In [ ]:
# Method to obtain measured and predicted cross section of an event sample corresponding
# to a (fiducial) kinematic subregion
def printCrossSection(weight_frame,desc,wnom_name="weights_nominal"):
    meas_xsec = np.sum(weight_frame[wnom_name])
    # sumw2 = np.sum(dataset[wnom_name]**2)
    # Neff = meas_xsec*meas_xsec/sumw2 => see nEff method
    print("\nFor {}, we have:".format(desc))
    print("  Measured x-sec:     {:.2f} fb".format(meas_xsec))
    print("  Effective statistics: {:.1f}".format(cu.nEff(weight_frame[wnom_name])))

print("\n\n===== FULL FIDUCIAL VOLUME =====")

printCrossSection(of_data,"nominal, MG5-based result, full fiducial phase space")
printCrossSection(of_data_hv,"Sherpa-based result, full fiducial phase space", wnom_name="weights_hv")

# Define a fiducial subset ("A" in equations above)
print("\n\n===== FIDUCIAL VOLUME pT(ll) > 500 GeV =====")

nominal_mc_mask = t["truth_pT_ll"].array(library="np") > 500
sherpa_mc_mask = t_sherpa["truth_pT_ll"].array(library="np") > 500


of_data_pt500 = of_data[nominal_mc_mask]
of_data_hv_pt500 = of_data_hv[sherpa_mc_mask]

printCrossSection(of_data_pt500,"nominal, MG5-based result, in region pT(ll) > 500 GeV")
printCrossSection(of_data_hv_pt500,"Sherpa-based result, in region pT(ll) > 500 GeV", wnom_name="weights_hv")

This shows how to produce the central value of the cross section measurement using the nominal MC sample, and the systematic varied value for the hidden variable unfolding uncertainty (produced with the alternative Sherpa MC sample).
The effective number of events (effective statistics) is calculated using $N_\text{eff} = \left(\sum_i w_i \right)^{2}/\sum_i w^2_i$.
Note that although there are over 1 million events in each sample, there are only 200k and 378k effective events for MadGraph and Sherpa respectively.

### Uncertainties

The measurement has a total of 31 sources of uncertainty. 

* One of these is the hidden variable unfolding uncertainty which is the difference between the measurement contructed with samples from different MC generators as discussed above.
* One uncertainty accounts for the data-driven unfolding uncertainty, which is calculated as the difference between the Madgraph MC sample binned with the weights `weights_dd` and `target_dd`.
* Three uncertainties account for stochastic effects and are stored as bootstrap variations: 
    * The data statistical uncertainty
    * The MC statistical uncertainty on the training set
    * The neural network stochastic variation
* One uncertainty accounts for the raw statistics of the nominal MC sample
* The remaining 25 uncertainties account for various systematic effects by varying the nominal event weights. The uncertainties are calculated by binning the Madgraph MC sample with these weights and taking the difference to the nominal measurement.

All of these uncertainties will be assessed and used to calculate the total uncertainty on the measured cross section below.

In [ ]:
# Lists of uncertainty variations
syst_nps = ["pileup","lumi","topBackground", # pileup modelling, luminosity, bkg subtraction (top). 3 NPs
            "nonstrongDiboson","nonstrongEW", # composition of non-strong processes in MC. 2 NPs
            "trackEffMain","trackEffJet","trackFake","trackPtScale", # track systematics. 4 NPs
            "muEffReco","muEffIso","muEffTrack","muEffTrig","muCalID","muCalMS","muCalResBias","muCalScale", # muon eff+calib. 8 NPs
            "theoryPSjet","theoryPSsoft","theoryMPI","theoryPSscale","theoryAlphaS","theoryQCD","theoryPDF"] # theory. 7 NPs

mc_train_stat  = [col for col in of_data.keys() if col.startswith("weights_bootstrap_mc")]
data_stat      = [col for col in of_data.keys() if col.startswith("weights_bootstrap_data")]
NN_stability   = [col for col in of_data.keys() if col.startswith("weights_ensemble")]

def printXsecUnc(dataset,desc,sherpa):
    meas_xsec  = np.sum(dataset["weights_nominal"])    # sum w
    Vstat_meas = np.sum(dataset["weights_nominal"]**2) # sum w^2, statisical variance

    Vmeas = 0 # total, fractional variance
    print("\n---------\n{}".format(desc))
    print("   Measured x-sec:   {:.2f} fb".format(meas_xsec))
    print("\n   Systematics:")
    for NP in syst_nps:
        syst_xsec = np.sum(dataset["weights_"+NP])
        unc = (syst_xsec/meas_xsec-1) # fractional uncertainty amplitude
        Vmeas += unc*unc
        print("  {:>15s}:    {:5.2f}%".format(NP,unc*100))
        
    # Measurments performed with alternative MC samples 
    meas_sherpa = np.sum(sherpa["weights_hv"]) # 'hidden variable uncertainty'
    meas_DD  = np.sum(dataset["weights_dd"]) # Data-driven unfolding uncertainty
    meas_DD_target  = np.sum(dataset["target_dd"])
    print("  {:>15s}:    {:5.2f}%".format("DD unfold. unc.",(meas_DD/meas_DD_target-1)*100))
    
    Vmeas += (meas_DD/meas_DD_target-1)**2
    Vmeas += (meas_sherpa/meas_xsec-1)**2
    print("  {:>15s}:    {:5.2f}% (HV unfold. unc.)".format("Sherpa meas",(meas_sherpa/meas_xsec-1)*100))
    print("  {:>15s}:    {:5.2f}%".format("Total syst",Vmeas**0.5*100))

    print("\n   Stochastic uncertainties:")
    print("  {:>15s}:    {:5.2f}%".format("MC stat",Vstat_meas**0.5/meas_xsec*100))
    Vmeas += Vstat_meas/meas_xsec/meas_xsec
    
    unc = np.std([np.sum(dataset[col]) for col in mc_train_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("MC train stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in data_stat])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("Data stat",unc*100))

    unc = np.std([np.sum(dataset[col]) for col in NN_stability])/meas_xsec
    Vmeas += unc*unc
    print("  {:>15s}:    {:5.2f}%".format("NN stability",unc*100))
    print("\n  {:>15s}:    {:5.2f}%".format("Total uncert",Vmeas**0.5*100))

In [ ]:
printXsecUnc(of_data,"Full fiducial phase space, i.e. pT(ll) > 200 GeV",of_data_hv)

In [ ]:
printXsecUnc(of_data_pt500,"pT(ll) > 500 GeV",of_data_hv_pt500)